# TetraFT — heal scale-up (0.8B on Kaggle)

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run (Qwen + transformers `qwen3_5`) |
| Flow | inventory → original PPL → shock PPL → QAFT |

**Locked heal DNA** (see `RESULTS.md`)
- c=0.25, absmean_channel, STE identity
- **`skip_linear_attn=True`** (GDN left FP)
- λ_warmup=**256**, peak lr 2e-4, **cosine → floor 0.1×**
- Disk-safe: weights-only best/final, `save_steps=0`

| Preset | ≈ tokens | Val PPL |
|--------|---------:|--------:|
| full_smoke + skip GDN | 5.2M | ~60.6 |
| **heal_25m** | **25M** | **~48.2** (done) |
| **heal_50m** | **50M** | **next — fresh start** |

**Sanity:** inventory ≈ **96 eligible / 91 skipped**, ~**41%** quantized.
Do **not** resume weights-only heal_25m as primary 50M path.

Logic in `run_smoke.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

# Fail fast if heal presets missing (stale tetraft-code)
from config import SMOKE_PRESETS
assert "heal_25m" in SMOKE_PRESETS, "heal_25m missing — refresh tetraft-code dataset"
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
import argparse
import shutil
from pathlib import Path

# --- Heal scale-up (c=0.25 locked; skip GDN via preset) ---
PRESET = "heal_50m"  # fresh 50M; heal_25m done (~48.2)
SAVE_OPTIMIZER = False  # True only for resume (large)
CLEAR_OUTPUT = True

OUTPUT_DIR = f"/kaggle/working/checkpoints_{PRESET}"

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=None,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=False,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,  # None = use preset (heal_* sets True)
    no_skip_linear_attn=False,
    seed=42,
    device_map="auto",
)
print(f"run preset={PRESET} save_optimizer={SAVE_OPTIMIZER} out={OUTPUT_DIR}")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    print("inventory", results["inventory_summary"])
    inv = results["inventory_summary"]
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear control — GDN skip may be off")
if "ppl_after_smoke" in results and "ppl_original" in results and results["ppl_original"]:
    print("after/orig =", results["ppl_after_smoke"] / results["ppl_original"])
print("targets: beat heal_25m ~48.2 @ 25M; trend toward orig ~17.7")

### Artifacts

Under `OUTPUT_DIR`:

- `linear_inventory.json` — expect ~96 eligible / 91 skipped
- `metrics.jsonl`, `smoke_results.json`
- `checkpoint-best`, `checkpoint-final` — weights-only

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| skip GDN scout | 5.2M | ~60.6 |
| **heal_25m** | **25M** | **~48.2** |
| heal_50m | 50M | **TBD** |

Record end PPL / after/orig in `RESULTS.md` when the job finishes.
